Bu notebook temizlenmiş kodları içermektedir. Eğitim çıktılarına ve performans grafiklerine projenin reports/ klasöründen veya README sayfasından ulaşabilirsiniz.

In [ ]:
# Yapay zeka modelimizin beyni olan kütüphaneyi yüklüyoruz
!pip install ultralytics

import ultralytics
ultralytics.checks() # GPU'nun aktif olup olmadığını bize söyleyecek

In [ ]:
# --- VERİ SETİ HAZIRLIĞI ---
# Eğer Google Colab kullanıyorsanız ve verileriniz Drive'daysa aşağıdaki satırları aktif edin:
# from google.colab import drive
# drive.mount('/content/drive')

# Veri setini çalışma alanına kopyalama (Örnek yol)
# !cp /content/drive/MyDrive/Yol/VeriSeti.zip .
# !unzip -q VeriSeti.zip -d datasets/

In [ ]:
import os

def veri_seti_ozeti(ana_dizin):
    klasorler = ['train', 'valid', 'test']
    print(f"{'Kkkkklasöööör2':<10} | {'Resiim Sayısı (.jpg)':<20} | {'Etiiket Sayısı (.txt)':<20}")
    print("-" * 55)

    for k in klasorler:
        yol = os.path.join(ana_dizin, k)
        if os.path.exists(yol):
            # Resimleri say (Genelde .jpg veya .png olur)
            resimler = [f for f in os.listdir(os.path.join(yol, 'images')) if f.endswith(('.jpg', '.jpeg', '.png'))]
            # Etiketleri say
            etiketler = [f for f in os.listdir(os.path.join(yol, 'labels')) if f.endswith('.txt')]

            print(f"{k:<10} | {len(resimler):<20} | {len(etiketler):<20}")
        else:
            print(f"{k:<10} | Klasör bulunamadı!")

# datasets klasörümüzü kontrol edelim
veri_seti_ozeti('datasets')

In [ ]:

from collections import Counter

def dosya_uzanti_say(ana_dizin):
    klasorler = ['train', 'valid', 'test']

    for k in klasorler:
        print(f"\n--- {k.upper()} Kkkkklasörü Analiziiiii 2 ---")
        img_yolu = os.path.join(ana_dizin, k, 'images')
        label_yolu = os.path.join(ana_dizin, k, 'labels')

        if os.path.exists(img_yolu):
            # Görüntü klasöründeki tüm uzantıları say
            img_uzantilar = [os.path.splitext(f)[1].lower() for f in os.listdir(img_yolu)]
            img_sayim = Counter(img_uzantilar)
            for uzanti, sayi in img_sayim.items():
                print(f"Görüntü Dosyası ({uzanti if uzanti else 'uzantısız'}): {sayi} adet")

        if os.path.exists(label_yolu):
            # Etiket klasöründeki tüm uzantıları say
            lbl_uzantilar = [os.path.splitext(f)[1].lower() for f in os.listdir(label_yolu)]
            lbl_sayim = Counter(lbl_uzantilar)
            for uzanti, sayi in lbl_sayim.items():
                print(f"Etiket Dosyası ({uzanti}): {sayi} adet")

dosya_uzanti_say('datasets')

In [ ]:
import yaml

# data.yaml dosyasını okuyup yemek isimlerini çekiyoruz
with open('datasets/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

yemekler = data['names']

print(f"--- Toplam Sınıff Sayısııı: {len(yemekler)} ---")
print("Hangi 40 tanesini seçelim??? İşte tam liste 2:")
print("-" * 35)

for i, isim in enumerate(yemekler):
    print(f"{i}: {isim}")

In [ ]:
import os
from collections import Counter

# Path ayarı
train_label_path = '/content/datasets/train/labels'

if not os.path.exists(train_label_path):
    print("HATA: Klasör bulunamadı!")
else:
    # 1. KONTROL: İlk 5 dosyadaki TÜM satır başlarını yazdır
    print("--- İLK 5 DOSYA DETAYLI SATIR BAŞI KONTROLÜ DÖRDÜNCÜ KERE ---")
    files = [f for f in os.listdir(train_label_path) if f.endswith('.txt')][:5]

    for f_name in files:
        print(f"\nDosya: {f_name}")
        ids_in_file = []
        with open(os.path.join(train_label_path, f_name), 'r') as f:
            for line_num, line in enumerate(f, 1):
                parts = line.split()
                if parts:
                    # Sadece satırın en başındaki ilk sayıyı alıyoruz
                    ids_in_file.append(parts[0])
        print(f"Bulunan Class ID'ler (Satır sırasıyla): {', '.join(ids_in_file)}")

    print("\n" + "="*50 + "\n")

    # 2. ANALİZ: En çok örneği olan 40 sınıfı bulalım
    print("Tüm veri seti analiz ediliyor (Instance sayımı)...")
    sayici = Counter()
    for file in os.listdir(train_label_path):
        if file.endswith('.txt'):
            with open(os.path.join(train_label_path, file), 'r') as f:
                for line in f:
                    parts = line.split()
                    if parts:
                        try:
                            # Satır başındaki değeri al (float ve int dönüşümü ile)
                            class_id = int(float(parts[0]))
                            sayici[class_id] += 1
                        except ValueError:
                            # Eğer sayı değilse (bozuk veriyse) buraya düşer
                            continue

    # En çok örneği olan ilk 40'ı bul
    en_iyi_40_tuple = sayici.most_common(40)
    en_iyi_40_listesi = [item[0] for item in en_iyi_40_tuple]

    print("--- ANALİZ SONUCU: EN ÇOK ÖRNEĞİ OLAN 40 SINIF ---")
    print(f"ID Listesi: {en_iyi_40_listesi}")
    print("\nDetaylı Liste (ID -> Toplam Nesne Sayısı):")
    for id_val, count in en_iyi_40_tuple:
        print(f"ID {id_val}: {count} adet")

In [ ]:
import os
import shutil

# --- AYARLAR ---
dataset_path = '/content/datasets'
output_path = '/content/food_40_clean'

# Görsellerinden okuduğum en güçlü 40 ID listesi (Sıralama korundu)
en_iyi_40 = [
    11, 96, 16, 24, 22, 82, 92, 74, 12, 45, 93, 52, 60, 80, 51, 73, 28, 10, 68, 69,
    14, 21, 20, 26, 36, 0, 83, 4, 9, 91, 1, 57, 62, 31, 61, 42, 19, 18, 70, 38
]

def filtrele_ve_kopyala(split):
    img_in = os.path.join(dataset_path, split, 'images')
    lab_in = os.path.join(dataset_path, split, 'labels')
    img_out = os.path.join(output_path, split, 'images')
    lab_out = os.path.join(output_path, split, 'labels')

    # Yeni klasörleri oluştur
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lab_out, exist_ok=True)

    print(f"🚀 {split} klasörü işleniyor, lütfen bekleyin...")
    sayac = 0

    for label_file in os.listdir(lab_in):
        if not label_file.endswith('.txt'): continue

        yeni_satirlar = []
        with open(os.path.join(lab_in, label_file), 'r') as f:
            for line in f:
                parts = line.split()
                if not parts: continue

                try:
                    # Sadece satırın başındaki ID'ye bakıyoruz
                    c_id = int(float(parts[0]))
                    if c_id in en_iyi_40:
                        # ID'yi yeni 0-39 sistemine çek, geri kalan koordinatları aynen ekle
                        yeni_id = en_iyi_40.index(c_id)
                        yeni_satirlar.append(f"{yeni_id} {' '.join(parts[1:])}\n")
                except: continue

        # Eğer seçilen 40 sınıftan birini bulduysak dosyaları kopyala
        if yeni_satirlar:
            # Yeni etiketi yaz
            with open(os.path.join(lab_out, label_file), 'w') as f:
                f.writelines(yeni_satirlar)

            # Resmi bul ve kopyala
            for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                img_name = label_file.replace('.txt', ext)
                if os.path.exists(os.path.join(img_in, img_name)):
                    shutil.copy2(os.path.join(img_in, img_name), os.path.join(img_out, img_name))
                    sayac += 1
                    break

    print(f"✅ {split} tamamlandııı! Toplam {sayac} set aktarıldı.")

# Başlat
for s in ['train', 'valid', 'test']:
    if os.path.exists(os.path.join(dataset_path, s)):
        filtrele_ve_kopyala(s)

print(f"\n✨ İŞLEM BİTTİ! Yeni veri setin tertemiz bir şekilde şurada: {output_path}")

In [ ]:
import os

def dolu_ve_bos_say(ana_dizin):
    # Ana dizinin varlığını kontrol et
    if not os.path.exists(ana_dizin):
        print(f"HATA: {ana_dizin} klasörü bulunamadı!")
        return

    klasorler = ['train', 'valid', 'test']
    print(f"{'Klasör':<10} | {'Toplam Resiiiim':<15} | {'Doluuuu (Etiketliiiii)':<15} | {'Boş (Arka Plan)':<15}")
    print("-" * 65)

    for k in klasorler:
        label_yolu = os.path.join(ana_dizin, k, 'labels')
        if os.path.exists(label_yolu):
            toplam = 0
            dolu = 0
            bos = 0

            # Sadece .txt dosyalarını listele
            label_dosyalari = [f for f in os.listdir(label_yolu) if f.endswith('.txt')]

            for dosya in label_dosyalari:
                toplam += 1
                yol = os.path.join(label_yolu, dosya)

                # Dosya boyutu 0'dan büyükse ve içinde veri varsa dolu say
                if os.path.getsize(yol) > 0:
                    dolu += 1
                else:
                    bos += 1

            print(f"{k:<10} | {toplam:<15} | {dolu:<15} | {bos:<15}")
        else:
            print(f"{k:<10} | Klasör yok!")

# Yeni temizlediğimiz klasörü kontrol ediyoruz
dolu_ve_bos_say('/content/food_40_clean')

In [ ]:
import yaml
import os

# 1. Orijinal isimleri içeren ana dosyayı oku
eski_yaml_yolu = '/content/datasets/data.yaml'
with open(eski_yaml_yolu, 'r') as f:
    eski_data = yaml.safe_load(f)
    # Eğer isimler listeyse (0, 1, 2...), direkt alıyoruz
    eski_isimler = eski_data['names']

# 2. Senin Remapping kodunda kullandığın liste (Sıralama ÇOK ÖNEMLİ)
en_iyi_40 = [
    11, 96, 16, 24, 22, 82, 92, 74, 12, 45, 93, 52, 60, 80, 51, 73, 28, 10, 68, 69,
    14, 21, 20, 26, 36, 0, 83, 4, 9, 91, 1, 57, 62, 31, 61, 42, 19, 18, 70, 38
]

# 3. Yeni isim listesini oluştur (0. sıraya 11'in ismi, 1. sıraya 96'nın ismi...)
# Bu sayede etiketlerdeki '0' artık yeni sistemdeki ilk yemeği temsil edecek.
yeni_isimler_listesi = []
for original_id in en_iyi_40:
    isim = eski_isimler[original_id]
    yeni_isimler_listesi.append(isim)

# 4. YAML içeriğini hazırla
yeni_yaml_content = {
    'path': '/content/food_40_clean',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 40,
    'names': yeni_isimler_listesi
}

# 5. Kaydet
yeni_yaml_adresi = '/content/food_40_clean/data.yaml'
with open(yeni_yaml_adresi, 'w') as f:
    yaml.dump(yeni_yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"✅ Yeni YAML baaaaaşarıyla oluşturuldu 2: {yeni_yaml_adresi}")
print(f"🚀 Yeni 0. sınıfın (Eski ID 11): {yeni_isimler_listesi[0]}")
print(f"🚀 Yeni 1. sınıfın (Eski ID 96): {yeni_isimler_listesi[1]}")

In [ ]:
from ultralytics import YOLO

# 1. Nano
model = YOLO('yolov8n-seg.pt')

# 2. Hız ve Doğruluk odaklı eğitim
results = model.train(
    data='/content/food_40_clean/data.yaml',
    epochs=50,              # 50 epoch hedef, patience (sabır) sayesinde erken durabilir
    imgsz=480,              # HIZ: 640'tan 480'e indik, süreyi %40 kısaltır
    batch=16,               # GÜVENLİ: Bellek hatası riskini sıfırlar
    patience=10,            # 10 tur boyunca mAP artmazsa eğitimi bitir (Zaman kazandırır)
    optimizer='AdamW',      # Modern ve hızlı yakınsama sağlayan optimizer
    workers=4,              # CPU'dan GPU'ya veri akışını hızlandırır
    device=0,               # T4 GPU kullanımı

    # --- mAP Artıran STRATEJİK Augmentation (Veri Artırma) ---
    mosaic=1.0,             # 4 resmi birleştirip küçük detayları (zeytin, bezelye vb.) öğretir
    degrees=15.0,           # Resimleri hafif döndürür (Tabak açısı farklarını çözer)
    flipud=0.5,             # Resimleri dikey çevirir (Yemeklerde çok etkilidir)
    hsv_s=0.7,              # Renk doygunluğu (Farklı ışık ve kamera kaliteleri için)
    translate=0.1,          # Resimleri hafif kaydırır

    # --- Teknik Çıktılar ---
    project='SmartPlate_AI',
    name='v2_mAP_Boost_Final',
    save=True,
    plots=True
)

In [ ]:
import shutil
from google.colab import files

# İndirmek istediğin klasörün yolu
folder_to_zip = '/content/runs/segment/SmartPlate_AI/v2_mAP_Boost_Final'

# Zip dosyasının adı
output_filename = 'SmartPlate_AI_v2_Egitim_Sonuclari'

# Klasörü zipleyelim
shutil.make_archive(output_filename, 'zip', folder_to_zip)

# Ziplenmiş dosyayı bilgisayarına indirelim
files.download(f'{output_filename}.zip')

print(f"✅ {output_filename}.zip oluşturuldu ve indirme başlatıldı!")